# Camada Gold

Camada final com objetivo criar agregações e métricas para responder as perguntas:

1) Existem diferenças relevantes de desempenho entre escolas públicas e privadas?
2) Qual a diferença de desempenho das escolas de diferentes niveis socioeconômicos?
3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?
4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?
5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?

# 1.0 Preparação 



## 1.1. Importação de bibliotecas e bases 
Importação das bibliotecas necessárias e leitura das bases a serem utilizadas provenientes das camadas silver e bronze

In [0]:
#Importação de bases 
from pyspark.sql.functions import when, col
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
#Leitura das 3 bases a serem utilizadas para gerar as métricas e agregações necessárias
df_saeb = spark.table("`mvp-eng-dados-puc-rio`.silver.resultados_saeb") #Base da camada silver tratada com resultados do SAEB
df_atributos_estados = spark.table("`mvp-eng-dados-puc-rio`.bronze.atributos_estados") # Base dimensão da camada bronze com a lista de estados e suas respectivas regiões
df_indicadores_estados = spark.table("`mvp-eng-dados-puc-rio`.bronze.dados_socioeconomicos_estados") #Base de fatos da camada bronze com indicadores socioeconômicos dos estados para diferentes censos do IBGE


In [0]:
#Seleção da camada gold para salvar as bases geradas

spark.sql("USE CATALOG `mvp-eng-dados-puc-rio`")
spark.sql("USE SCHEMA `gold`")


In [0]:
#visualização de amostra da base df_saeb
display(df_saeb.limit(10))

In [0]:
#visualização de amostra da base df_atributos_estados

display(df_atributos_estados.limit(10))

In [0]:
#visualização de amostra da base df_indicadores_estados

display(df_indicadores_estados.limit(10))

## 1.2. Filtragem das tabelas

Com o objetivo de preparar as tabelas para análise, alguns filtros serão feitos nas bases da camada silver e bronze utilizadas




### 1.2.1. Filtragem da base df_saeb

Foi definido para a análise a avaliação dos dados do último SAEB para o 9º ano do ensino fundamental, para isso foram feitos os filtros:

1) Filtrar a base com o objetivo de utilizar somente o último SAEB para geração das tabelas da camada gold. Na base original só há o resultado do SAEB de 2023, mas esse filtro deixará a camada gold preparada para analisar o resultado de novos SAEBs, caso seja feito o upload desses dados
2) Filtrar somente escolas que foram avaliadas no 9º ano do ensino fundamental

In [0]:
1# 1 e 2) Filtrar escolas avaliadas no 9º EF e último SAEB
# ============================================================

# Obtenção dos registros do último SAEB realizando um filtro na coluna ID_SAEB a partir do seu maior valor com resultados para o 9º ano do ensino fundamental
max_id_saeb = (
    df_saeb
    .filter(F.col("Escola_Avaliada_9EF") == "Sim")
    .agg(F.max("ID_SAEB").alias("max_id_saeb"))
    .collect()[0]["max_id_saeb"]
)

df_saeb_filtrado  = (
    df_saeb
    .filter(
        (F.col("Escola_Avaliada_9EF") == "Sim") &
        (F.col("ID_SAEB") == max_id_saeb)
    )
)


In [0]:
#célula criada para avaliar se o filtro realizado funcionou

qtd_antes = df_saeb.count()
qtd_depois = df_saeb_filtrado.count()
print(f"quantidade antes: {qtd_antes}")
print(f"quantidade depois: {qtd_depois}")


Como pode ser visto, nossa base diminuiu, mantendo somente as escolas avaliadas no 9º ano do ensino fundamental

### 1.2.2. Filtragem da base de indicadores socioeconômicos

Nesta base foi feito um filtro para somente pegar os indicadores do último censo do IBGE disponível


In [0]:
# ============================================================
#Filtrar df_indicadores_estados pelo maior ANO de realização do censo do IBGE
# ============================================================

max_ano = (
    df_indicadores_estados
    .agg(F.max("ANO").alias("max_ano"))
    .collect()[0]["max_ano"]
)

df_indicadores_filtrado = (
    df_indicadores_estados
    .filter(F.col("ANO") == max_ano)
)

In [0]:
#célula criada para avaliar se o filtro realizado funcionou

qtd_antes = df_indicadores_estados.count()
qtd_depois = df_indicadores_filtrado.count()
print(f"quantidade antes: {qtd_antes}")
print(f"quantidade depois: {qtd_depois}")
df_indicadores_filtrado

Como pode ser visto acima sobraram 27 registros (1 por estado) mostrando que a filtragem funcionou

# 2. Geração de tabela para responder se existem diferenças relevantes de desempenho entre escolas públicas e privadas

Para isso, será feita uma tabela que faça o agrupamento por tipo de escola e calcule as médias de português e matemática

In [0]:
#Agrupamento pelo tipo de escola (escola_publica), contagem da quantidade de escola por tipo e média das notas de português e matemática

df_resumo_publico_privada = (
    df_saeb_filtrado
    .groupBy("escola_publica")
    .agg(
        F.count("*").alias("quantidade_escolas"),
        F.avg("MEDIA_9EF_LP").alias("media_9EF_LP"),
        F.avg("MEDIA_9EF_MT").alias("media_9EF_MT")
    )
    .orderBy("escola_publica")
)

display(df_resumo_publico_privada)

Como só há escolas públicas, não é possível avaliar o resultado entre públicas e privadas, entretanto, se no futuro forem adicionadas, será gerada uma tabela com essa informação

## 2.1. Salvar base na camada gold

In [0]:
df_resumo_publico_privada.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_tipo_de_escola")

# 3. Geração de tabela por nível socioeconômico para responder a pergunta 2
(Qual a diferença de desempenho das escolas de diferentes niveis socioeconômicos?)

Para responder essa pergunta, é preciso explicar as classificações do nível socioeconômico

- **Nível I** : O estrato mais baixo; famílias com menor escolaridade (até o fundamental incompleto) e posse restrita de bens básicos (como geladeira, TV e celular).
- **Níveis II, III e IV**: Indicam faixas de transição com aumento gradual de itens de conforto em casa e anos de estudo dos responsáveis.
- **Níveis V e VI**: Concentram a maior parte das escolas e estudantes brasileiros, com escolaridade média no ensino fundamental/médio completo e maior acesso a eletrodomésticos e tecnologia.
- **Níveis VII e VIII**: Os patamares mais elevados da escala, caracterizados por maior escolaridade parental (ensino superior) e maior infraestrutura de bens e serviços contratados no domicílio.

Para fazer essa avaliação, foram adotadas duas abordagens a serem geradas em uma tabela unificada.

1) Foi calculada a média de proficiência por nível socioeconômico para identificar diferenças de desempenho entre os grupos. Como as notas do SAEB são baseadas na TRI, também foram criadas categorias de proficiência para facilitar a interpretação dos resultados.

2) Para aprofundar a análise, foram calculados os percentuais de escolas em cada faixa de proficiência — Abaixo do Básico, Básico, Adequado e Avançado — por nível socioeconômico. As quatro categorias totalizam 100% das escolas de cada grupo, permitindo analisar não apenas a média, mas também como as escolas se distribuem entre as diferentes faixas de desempenho. O mesmo foi realizado para matemática

## 3.1. Tratamento da Base

In [0]:
# Pega automaticamente todas as categorias existentes da coluna PROFICIENCIA_9EF_LP no df_saeb_filtrado (português)
categorias_lp = [
    r["PROFICIENCIA_9EF_LP"]
    for r in df_saeb_filtrado
        .select("PROFICIENCIA_9EF_LP")
        .distinct()
        .collect()
    if r["PROFICIENCIA_9EF_LP"] is not None
]


# Pega automaticamente todas as categorias existentes da coluna PROFICIENCIA_9EF_MT no df_saeb_filtrado (matemática)

categorias_mt = [
    r["PROFICIENCIA_9EF_MT"]
    for r in df_saeb_filtrado
        .select("PROFICIENCIA_9EF_MT")
        .distinct()
        .collect()
    if r["PROFICIENCIA_9EF_MT"] is not None
]

# Monta as agregações básicas de média
agregacoes = [
    F.count("*").alias("quantidade_escolas"),
    F.round(F.avg("MEDIA_9EF_LP"), 2).alias("media_9EF_LP"),
    F.round(F.avg("MEDIA_9EF_MT"), 2).alias("media_9EF_MT")
]

# Geração de Percentuais de cada categoria de LP
for categoria in categorias_lp: #interação para gerar todas as colunas
    nome_coluna = f"{categoria}_LP"  #criação de coluna de LP

    agregacoes.append(   #calculo da percentagem da categoria gerada
        F.round(
            F.sum(
                F.when(
                    F.col("PROFICIENCIA_9EF_LP") == categoria, 1
                ).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias(nome_coluna)
    )

# Percentuais de cada categoria de MT
for categoria in categorias_mt:
    nome_coluna = f"{categoria}_MT"

    agregacoes.append(  #calculo da percentagem da categoria gerada
        F.round(
            F.sum(
                F.when(
                    F.col("PROFICIENCIA_9EF_MT") == categoria, 1
                ).otherwise(0)
            ) / F.count("*") * 100,
            2
        ).alias(nome_coluna)
    )

# Agrupamento final  #realização do agrupamento
df_resumo_socioeconomico = (
    df_saeb_filtrado
    .groupBy("NIVEL_SOCIO_ECONOMICO")
    .agg(*agregacoes)
    .orderBy("NIVEL_SOCIO_ECONOMICO")
)

# ------------------------------------------------------------
# Classificação de lingua portuguesa para o 9º ano do ensino fundamental para maior clareza da média do nível
# ------------------------------------------------------------
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumn(
    "PROFICIENCIA_9EF_LP",
    when(col("MEDIA_9EF_LP").isNull(), "Sem_resultado")
    .when(col("MEDIA_9EF_LP") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_LP") <= 275, "Básico")
    .when(col("MEDIA_9EF_LP") <= 325, "Adequado")
    .otherwise("Avançado")
)


# ------------------------------------------------------------
# Classificação de matemática para o 9º ano do ensino fundamental para maior clareza da média do nível
# ------------------------------------------------------------
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumn(
    "PROFICIENCIA_9EF_MT",
    when(col("MEDIA_9EF_MT").isNull(), "Sem_resultado")
    .when(col("MEDIA_9EF_MT") <= 225, "Abaixo do Básico")
    .when(col("MEDIA_9EF_MT") <= 300, "Básico")
    .when(col("MEDIA_9EF_MT") <= 350, "Adequado")
    .otherwise("Avançado")
)


#reorganização das colunas com o objetivo de facilitar a visualização
df_resumo_socioeconomico = df_resumo_socioeconomico.select(
    # Identificação
    "NIVEL_SOCIO_ECONOMICO",
    "quantidade_escolas",

    # Língua Portuguesa
   "PROFICIENCIA_9EF_LP",
    "media_9EF_LP",
    "Abaixo do Básico_LP",
    "Básico_LP",
    "Adequado_LP",
    "Avançado_LP",

    # Matemática
    "PROFICIENCIA_9EF_MT",
    "media_9EF_MT",
    "Abaixo do Básico_MT",
    "Básico_MT",
    "Adequado_MT",
    "Avançado_MT"
)

#Ajuste de nomenclatura de coluna para salvar no databricks
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumnRenamed("Abaixo do Básico_MT", "Abaixo_do_Básico_MT")
df_resumo_socioeconomico = df_resumo_socioeconomico.withColumnRenamed("Abaixo do Básico_LP", "Abaixo_do_Básico_LP")


display(df_resumo_socioeconomico)

##3.2. Salvando a base na camada gold

In [0]:
df_resumo_socioeconomico.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_nivel_socioeconomico")


## 3.3. Conclusões:

1) Dos 7 niveis avaliados:
-  **6** apresentam um nível **básico** de proficiência para português e matemática, 
- somente o **nível 7**, com maior escolaridade parental e infraestrutura **apresentou um nível adequado**.

2) **Avaliando as médias**, é possível identificar excluindo o nível 1 que apresenta uma amostra pequena (somente 23 escolas) que:
-  há uma **clara correlação entre maior nível socioeconômico e maior notas do SAEB**

2) Entretanto, Apesar de **6 dos 7 níveis apresentarem nota média**  para português e matemática num nível básico, eles **estão em faixas muito diferentes dessa escala**, com:
-  o nível 2 estando 2 pontos acima do corte inferior do nível básico ( média 227 vs corte 225)
- o **nível 6 estando com a média a menos de 1 ponto de chegar ao nível adequado (média 274 vs corte do nível adequado de 275).**

3) Esse ponto é fortalecido pelas colunas geradas de proficiência, com:
-  o **nível 2** apresentando  **mais da metade das escolas com níveis abaixo do básico para português (51%) e matemática (52%)**
- Esse indice que cai para **menos de 1% no nível 6**

# 4. Geração de tabela por estado para resposta das perguntas 3, 4 e 5

3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?
4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?
5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?


Para facilitar a resposta a esse tipo de perguntas é necessário gerar uma tabela unificada com os desempenhos médios por estado no SAEB e com os indicadores socioeconômicos por estado. Para isso, os seguintes passos foram realizados:


4.1) Agrupamento dos resultados do saeb por escola para português e matemática

4.2.)Realização de join com a tabela dimensão df_atributos para obter dados gerais dos estados (obs: criada etapa 4.2.1 para avaliar a qualidade desse join, verificando se houve multiplicação de linhas, por exemplo)

4.3.) Realização de join com a tabela dimensão df_indicadores para obter os indicadores socioeconômicos dos estados (obs: criada etapa 4.3.1 para avaliar a qualidade desse join, verificando se houve multiplicação de linhas, por exemplo)

4.4.) Para visualmente facilitar a análise foram criados dois rankings, um pelo indicador de IDHM que foi o escolhido para a análise e outro fazendo o valor médio entre as notas de português e matemática para cada estado. Com isso, a comparação passará a ser feita a partir de posição entre os estados, facilitando a visualização

4.5.) Além disso, foi feita uma etapa para separar esses 2 rankings em quartis e criada uma métrica para avaliar se o estado está no mesmo quartil nos dois rankings ou se está em ranking diferentes, permitindo rapidamente identificar quem está acima ou abaixo do esperado, ajudando a responder as perguntas 4 e 5

## 4.1. Agrupamento dos resultados do SAEB por escola para português e matemática

Calculado a partir da média dessas duas disciplinas

In [0]:
df_saeb_estado = (
    df_saeb_filtrado
    .groupBy("ESTADO")
    .agg(
        F.avg("MEDIA_9EF_LP").alias("MEDIA_9EF_LP"),
        F.avg("MEDIA_9EF_MT").alias("MEDIA_9EF_MT")
    )
)

In [0]:
display(df_saeb_estado.limit(10))

## 4.2. Join com a tabela df_atributos_estados

Utilizando pela parte da tabela de resultados do saeb a coluna ESTADO como chave e na df_atributos_estados a coluna Estado. Usado left join para não perder registros de estados da tabela saeb em caso de algum erro na df_atributos_estados

In [0]:
# ============================================================
# 4) LEFT JOIN com df_atributos_estados
# ============================================================

df_saeb_atributos = (
    df_saeb_estado.alias("saeb")
    .join(
        df_atributos_estados.alias("atrib"),
        F.col("saeb.ESTADO") == F.col("atrib.Estado"),
        "left"
    )
    .select(
        F.col("saeb.ESTADO"),
        F.col("saeb.MEDIA_9EF_LP"),
        F.col("saeb.MEDIA_9EF_MT"),
        F.col("atrib.Região")
    )
)

In [0]:
display(df_saeb_atributos.limit(10))

###4.2.1. Qualidade da transformação

Foram feitas checagens para verificar se o quantitativo de linhas antes é igual a depois do join. 

In [0]:
# ============================================================
# 5) Checagem do primeiro JOIN
# ============================================================

# Quantidade de linhas antes e depois
qtd_antes_join_1 = df_saeb_estado.count()
qtd_depois_join_1 = df_saeb_atributos.count()

print("=== CHECAGEM JOIN 1 ===")
print(f"Linhas antes do join:  {qtd_antes_join_1}")
print(f"Linhas depois do join: {qtd_depois_join_1}")

if qtd_depois_join_1 > qtd_antes_join_1:
    print("ATENÇÃO: houve multiplicação de linhas no primeiro join.")
else:
    print("OK: não houve multiplicação de linhas.")


# Estados sem correspondência na dimensão
sem_join_1 = (
    df_saeb_atributos
    .filter(F.col("Região").isNull())
    .select("ESTADO")
    .distinct()
)

qtd_sem_join_1 = sem_join_1.count()

print(f"Estados sem correspondência em df_atributos_estados: {qtd_sem_join_1}")

if qtd_sem_join_1 > 0:
    print("Estados sem correspondência:")
    sem_join_1.show(truncate=False)
else:
    print("OK: todos os estados encontraram correspondência.")


# Checagem adicional:
# Verificar se a chave Estado é realmente única na dimensão
duplicados_atributos = (
    df_atributos_estados
    .groupBy("Estado")
    .count()
    .filter(F.col("count") > 1)
)

qtd_duplicados_atributos = duplicados_atributos.count()

print(f"Estados duplicados na dimensão: {qtd_duplicados_atributos}")

if qtd_duplicados_atributos > 0:
    duplicados_atributos.show(truncate=False)

##4.3. Join com a tabela de indicadores socioeconômicos

utlizando a coluna ESTADO no dataframe que vinha sendo trabalho (df_saeb_atributos) e UFN na df_indicadores_filtrado

In [0]:
# ============================================================
# 7) LEFT JOIN com df_indicadores_estados
#    df_saeb_atributos.Estado → df_indicadores_filtrado.UFN
# ============================================================

df_final = (
    df_saeb_atributos.alias("saeb")
    .join(
        df_indicadores_filtrado.alias("ind"),
        F.col("saeb.ESTADO") == F.col("ind.UFN"),
        "left"
    )
    .select(
        # Todas as colunas de df_saeb_atributos
        *[F.col(f"saeb.{c}") for c in df_saeb_atributos.columns],

        # Todas as colunas de df_indicadores_filtrado, exceto UFN
        *[
            F.col(f"ind.{c}")
            for c in df_indicadores_filtrado.columns
            if c != "UFN"
        ]
    )
)



In [0]:
display(df_final.limit(30))

### 4.3.1. Qualidade dos dados

Checagens para verificar se houve duplicações/perdas de linhas ou não

In [0]:
# ============================================================
# 8) Checagem do segundo JOIN
# ============================================================

qtd_antes_join_2 = df_saeb_atributos.count()
qtd_depois_join_2 = df_final.count()

print("=== CHECAGEM JOIN 2 ===")
print(f"Linhas antes do join:  {qtd_antes_join_2}")
print(f"Linhas depois do join: {qtd_depois_join_2}")

if qtd_depois_join_2 > qtd_antes_join_2:
    print("ATENÇÃO: houve multiplicação de linhas no segundo join.")
else:
    print("OK: não houve multiplicação de linhas.")


# Verificar ESTADOS sem correspondência em UFN
sem_join_2 = (
    df_saeb_atributos
    .select("ESTADO")
    .distinct()
    .join(
        df_indicadores_filtrado
        .select("UFN")
        .distinct(),
        F.col("ESTADO") == F.col("UFN"),
        "left_anti"
    )
)

qtd_sem_join_2 = sem_join_2.count()

print(
    f"Estados sem correspondência em "
    f"df_indicadores_estados: {qtd_sem_join_2}"
)

if qtd_sem_join_2 > 0:
    print("Estados sem correspondência:")
    sem_join_2.show(truncate=False)
else:
    print("OK: todos os Estados encontraram correspondência.")

## 4.4. Criação dos rankings


Ranking de notas criado a partir de uma nota unificada dada por:  (nota de matemática + nota de português)/2

Ranking socioeconômico criado a partir do indicador IDHM, que foi o selecionado para essa análise, ranqueando do maior IDH até o menor

In [0]:
# ============================================================
# 9) Criação dos rankings
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window



# ------------------------------------------------------------
# Ranking de IDHM
# ------------------------------------------------------------

# Ranking: maior IDHM = maior posição no ranking
window_idhm = Window.orderBy(
    F.col("IDHM").desc()
)

df_final = (
    df_final
    .withColumn(
        "RANKING_IDHM",
        F.dense_rank().over(window_idhm)
    )
)

# ------------------------------------------------------------
# Ranking de desempenho no SAEB - 9º EF
# ------------------------------------------------------------




# Média unificada das proficiências de Língua Portuguesa
# e Matemática
df_final = (
    df_final
    .withColumn(
        "MEDIA_SAEB_9EF",
        (
            F.col("MEDIA_9EF_LP") +
            F.col("MEDIA_9EF_MT")
        ) / 2
    )
)

# Ranking: maior média de proficiência = melhor posição
window_saeb = Window.orderBy(
    F.col("MEDIA_SAEB_9EF").desc()
)
df_final = (
    df_final
    .withColumn(
        "RANKING_SAEB_9EF",
        F.dense_rank().over(window_saeb)
    )
)


In [0]:
display(df_final.limit(30))

## 4.5. Criação dos quartis e métrica CLASSIFICACAO_IDHM_SAEB

Os quartis foram gerados para o ranking de notas e para o de IDHM. E

Para os quartis tanto de notas quanto de IDHM, quanto menor o valor, melhor o resultado, logo:

 Quartil 1 = estados com maiores médias no SAEB

 Quartil 4 = estados com menores médias no SAEB


Em seguida, foi criada a métrica CLASSIFICACAO_IDHM_SAEB que fez a seguinte categorização:

1) Caso os quartis sejam iguais: Em linha com o esperado
2) Caso o quartil de nota seja 1 acima do quartil de IDHM: Levemente acima do esperado
3) Caso o quartil de nota seja 2 acima do quartil de IDHM: Acima do esperado
4) Caso o quartil de nota seja 3 acima do quartil de IDHM: Muito acima do esperado

2) Caso o quartil de nota seja 1 abaixo do quartil de IDHM: Levemente abaixo do esperado
3) Caso o quartil de nota seja 2 abaixo do quartil de IDHM: Abaixo do esperado
4) Caso o quartil de nota seja 3 abaixo do quartil de IDHM: Muito abaixo do esperado

In [0]:
# ============================================================
# 10) CRIAÇÃO DOS QUARTIS
# ============================================================

# ------------------------------------------------------------
# 10.1) Quartil do SAEB
# ------------------------------------------------------------
# Quartil 1 = estados com maiores médias no SAEB
# Quartil 4 = estados com menores médias no SAEB

window_quartil_saeb = Window.orderBy(
    F.col("MEDIA_SAEB_9EF").desc()
)

df_final = (
    df_final
    .withColumn(
        "QUARTIL_SAEB",
        F.ntile(4).over(window_quartil_saeb)
    )
)


# ------------------------------------------------------------
# 10.2) Quartil do IDHM
# ------------------------------------------------------------
# Quartil 1 = estados com maiores IDHM
# Quartil 4 = estados com menores IDHM

window_quartil_idhm = Window.orderBy(
    F.col("IDHM").desc()
)

df_final = (
    df_final
    .withColumn(
        "QUARTIL_IDHM",
        F.ntile(4).over(window_quartil_idhm)
    )
)


# ============================================================
# 11) DIFERENÇA ENTRE OS QUARTIS
# ============================================================

df_final = (
    df_final
    .withColumn(
        "DIF_QUARTIS",
        F.abs(
            F.col("QUARTIL_SAEB") -
            F.col("QUARTIL_IDHM")
        )
    )
)


# ============================================================
# 12) CLASSIFICAÇÃO DA RELAÇÃO IDHM x SAEB
# ============================================================
#
# Quartil SAEB menor que quartil IDHM:
# → posição relativa do SAEB é melhor
#
# Quartil SAEB maior que quartil IDHM:
# → posição relativa do SAEB é pior
#
# Mesmos quartis:
# → resultados em linha
# ============================================================

df_final = (
    df_final
    .withColumn(
        "CLASSIFICACAO_IDHM_SAEB",

        # Mesmo quartil
        F.when(
            F.col("DIF_QUARTIS") == 0,
            "Em linha com o esperado"
        )

        # SAEB 1 quartil acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 1),
            "Levemente acima do esperado"
        )

        # SAEB 2 quartis acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 2),
            "Acima do esperado"
        )

        # SAEB 3 quartis acima
        .when(
            (F.col("QUARTIL_SAEB") < F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 3),
            "Muito acima do esperado"
        )

        # SAEB 1 quartil abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 1),
            "Levemente abaixo do esperado"
        )

        # SAEB 2 quartis abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 2),
            "Abaixo do esperado"
        )

        # SAEB 3 quartis abaixo
        .when(
            (F.col("QUARTIL_SAEB") > F.col("QUARTIL_IDHM")) &
            (F.col("DIF_QUARTIS") == 3),
            "Muito abaixo do esperado"
        )
    )
)

### 4.5.1. Salvando a base na camada gold

Base salva na camada gold pronta para análises. Mantidos demais indicadores

In [0]:
# seleção das colunas de interesse para a análise de IDHM

df_final = (df_final.select(
    "ESTADO",
    "MEDIA_9EF_LP",
    "MEDIA_9EF_MT",
    "MEDIA_SAEB_9EF",
    "IDHM",
    "RANKING_SAEB_9EF",
    "RANKING_IDHM",
    "QUARTIL_SAEB",
    "QUARTIL_IDHM",
    "DIF_QUARTIS",
    "CLASSIFICACAO_IDHM_SAEB"
).orderBy(
    "DIF_QUARTIS",
    F.col("MEDIA_SAEB_9EF").desc()))

In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("resultado_saeb_por_estado_vs_idhm")

## 4.6. Visualização e Análise

In [0]:
# ============================================================
# 13) VISUALIZAÇÃO DO RESULTADO
# ============================================================

display(df_final.select(
    "ESTADO",
    "MEDIA_9EF_LP",
    "MEDIA_9EF_MT",
    "MEDIA_SAEB_9EF",
    "IDHM",
    "RANKING_SAEB_9EF",
    "RANKING_IDHM",
    "QUARTIL_SAEB",
    "QUARTIL_IDHM",
    "DIF_QUARTIS",
    "CLASSIFICACAO_IDHM_SAEB"
).orderBy(
    "DIF_QUARTIS",
    F.col("MEDIA_SAEB_9EF").desc())
.limit(30))

In [0]:
# Cálculo da correlação entre IDHM e a nota média do SAEB
corr = df_final.stat.corr("IDHM", "MEDIA_SAEB_9EF", method="pearson")
print(f"Correlação: {corr}")

**3) Quais estados apresentam os melhores e piores desempenhos médios no SAEB?**

A partir da tabela gerada acima, os estados com melhores desempenhos são Ceará, Paraná, Goiás e Santa Catarina

**4) Quais estados entregam um desempenho educacional abaixo do esperado para seu nível de desenvolvimento?**

Tem dois estados que apresentam resultados abaixo do esperado:
- O Amapá, que está em 12º no ranking de IDHM, mas na 25º posição no SAEB
- Roraima, em 13º no ranking de IDHM e na última posição do SAEB

**5) Existem estados que se destacam por apresentar bons resultados educacionais mesmo possuindo indicadores socioeconômicos inferiores?**

Sim, os destaques são:
- O estado do Ceará, com primeiro lugar no ranking do SAEB e 17º lugar no ranking do IDHM
- O estado de Alagoas, com o 8º lugar no ranking do SAEB e 26 no de IDHM


OBS: Foi também calculada a correlação entre IDHM e a nota do SAEB, encontrando um valor de 0.52, que é uma correlação linear positiva moderada.
